In [1]:
"""
@author: Zilan Cheng
@note: This code is modified based on Zongyi Li's original implementation of Fourier Neural Operators.
"""

import torch.nn.functional as F
from timeit import default_timer
from utilities3 import *
import numpy as np
import matplotlib.pyplot as plt
torch.cuda.set_device(0)
torch.manual_seed(0)
np.random.seed(0)

In [2]:
class SpectralConv2d(nn.Module):
    def __init__(self, width, modes,basis):
        super(SpectralConv2d, self).__init__()

        """
        2D integration layer. It does POD transform, linear transform, and POD transform.    
        """

        self.width=width
        self.modes = modes
        self.basis = basis.to(torch.float32)[:,:,:self.modes].to(device)

        self.scale = (1 / (self.width*self.width))
        self.weights1 = nn.Parameter(self.scale * torch.rand(self.width, self.width, self.modes, dtype=torch.float))

    def forward(self, x):
        batchsize = x.shape[0]
        x_1  = torch.einsum("bwxy,xyf->bwf",x,self.basis).to(device) # f refers to coefficients space,  this step projects from physical space to coefficients space
        x_1 = torch.einsum("bif,iof->bof",x_1,self.weights1).to(device) # this step defines a diagonal operator 
        x_out  = torch.einsum("bwf,xyf->bwxy",x_1,self.basis).to(device) # f refers to coefficients space, this step recovers from coefficients space to physical space
        return x_out


In [3]:
class MLP(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels):
        super(MLP, self).__init__()
        self.mlp1 = nn.Conv2d(in_channels, mid_channels, 1)
        self.mlp2 = nn.Conv2d(mid_channels, out_channels, 1)

    def forward(self, x):
        x = self.mlp1(x)
        x = F.gelu(x)
        x = self.mlp2(x)
        return x

In [4]:
def get_grid(shape, device):
    batchsize, size_x, size_y = shape[0], shape[1], shape[2]
    gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
    gridx = gridx.reshape(1, size_x, 1, 1).repeat([batchsize, 1, size_y, 1])
    gridy = torch.tensor(np.linspace(0, 1, size_y), dtype=torch.float)
    gridy = gridy.reshape(1, 1, size_y, 1).repeat([batchsize, size_x, 1, 1])
    return torch.cat((gridx, gridy), dim=-1).to(device)

In [5]:
class PODNO2d(nn.Module):
    def __init__(self, modes,width,basis):
        super(PODNO2d, self).__init__()

        """
        The overall network. 
        1. Lift the input to the desire channel dimension.
        2. 4 integration layers.
        3. Project back to the physical space.
        
        input: (a(x, y), x, y)
        output: u(x,y)
        """

        self.modes = modes
        self.width = width
        self.basis = torch.tensor(basis)
        
        self.p = nn.Linear(3, self.width)
        self.conv0 = SpectralConv2d(self.width, self.modes, self.basis)
        self.conv1 = SpectralConv2d(self.width, self.modes, self.basis)
        self.conv2 = SpectralConv2d(self.width, self.modes, self.basis)
        self.conv3 = SpectralConv2d(self.width, self.modes, self.basis)
        self.mlp0 = MLP(self.width, self.width, self.width)
        self.mlp1 = MLP(self.width, self.width, self.width)
        self.mlp2 = MLP(self.width, self.width, self.width)
        self.mlp3 = MLP(self.width, self.width, self.width)
        self.w0 = nn.Conv2d(self.width, self.width, 1)
        self.w1 = nn.Conv2d(self.width, self.width, 1)
        self.w2 = nn.Conv2d(self.width, self.width, 1)
        self.w3 = nn.Conv2d(self.width, self.width, 1)
        self.q = MLP(self.width, 1, self.width * 4) 

    def forward(self, x):
        grid = get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x=x.to(torch.float32)
        x = self.p(x)
        x = x.permute(0, 3, 1, 2)
        x1 = self.conv0(x)
        x1 = self.mlp0(x1)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x1 = self.mlp1(x1)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x1 = self.mlp2(x1)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x1 = self.mlp3(x1)
        x2 = self.w3(x)
        x = x1 + x2

        x = self.q(x)
        x = x.permute(0, 2, 3, 1)
        return x

In [6]:
################################################################
# configs
################################################################
ntrain = 900
ntest = 100
nsnap=900

width = 32
modes = 36

s = 256
r = 2

batch_size = 20
learning_rate = 0.001
epochs = 2000
iterations = epochs*(ntrain//batch_size)

In [7]:
################################################################
# dataloader
################################################################
u_end=np.load("../data/kp/u_end_512.npy")
u_end=torch.tensor(u_end)
u_end=u_end.permute(2,0,1)
u1=np.load("../data/kp/u1_512.npy")
u1=torch.tensor(u1)
u1=u1.permute(2,0,1)

x_train=u1[:ntrain,:,:][:,::r,::r]
x_test=u1[-ntest:,:,:][:,::r,::r]
y_train=u_end[:ntrain,:,:][:,::r,::r]
y_test=u_end[-ntest:,:,:][:,::r,::r]

x_train = x_train.reshape(ntrain,s,s,1)
x_test = x_test.reshape(ntest,s,s,1)
y_train = y_train.reshape(ntrain,s,s,1)
y_test = y_test.reshape(ntest,s,s,1)

x_normalizer = UnitGaussianNormalizer(x_train)
x_train = x_normalizer.encode(x_train)
x_test = x_normalizer.encode(x_test)

y_normalizer = UnitGaussianNormalizer(y_train)
y_train = y_normalizer.encode(y_train)

train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test, y_test), batch_size=batch_size, shuffle=False)

In [8]:
################################################################
# SVD
################################################################
x_snap = x_train[:nsnap,:,:,:]
y_snap = y_train[:nsnap,:,:,:]
basis_0=torch.cat((x_snap,y_snap),0)
basis_1=basis_0.permute(3,1,2,0)
basis_1=basis_1.reshape(-1,2*nsnap)
U,S,V=torch.svd(basis_1)
basis=U.reshape(s,s,-1)

In [9]:
solution_real=torch.zeros(ntest,s,s,1)
solution_learnt=torch.zeros(ntest,s,s,1)

In [10]:
################################################################
# training and evaluation
################################################################
model = PODNO2d(modes, width,basis).to(device) 
print(count_params(model))

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=iterations)

myloss = LpLoss(size_average=True)
y_normalizer.to(device)
for ep in range(epochs):
    i=0
    model.train()
    train_l2 = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x).reshape(batch_size, s, s,1)
        out = y_normalizer.decode(out)
        y = y_normalizer.decode(y)

        loss = myloss(out.view(batch_size,-1), y.view(batch_size,-1))
        loss.backward()

        optimizer.step()
        scheduler.step()
        train_l2 += loss.item()
    model.eval()
    test_l2 = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            out = model(x).reshape(batch_size, s, s,1)
            out = y_normalizer.decode(out)
            solution_real[i:i+batch_size,:,:,:]=y
            solution_learnt[i:i+batch_size,:,:,:]=out
            i=i+batch_size
            test_l2 += myloss(out.view(batch_size,-1), y.view(batch_size,-1)).item()
    train_l2/= ntrain/batch_size
    test_l2 /= ntest/batch_size
    
    if ep % 50 == 0 or ep == epochs - 1:
        print(f"Epoch {ep:4d} | Train L2: {train_l2:.6f} | Test L2: {test_l2:.6f}")

/tmp/ipykernel_3164926/4233491345.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.basis = torch.tensor(basis)


164609
Epoch    0 | Train L2: 0.366118 | Test L2: 0.287220
Epoch   50 | Train L2: 0.041552 | Test L2: 0.045979
Epoch  100 | Train L2: 0.034758 | Test L2: 0.039089
Epoch  150 | Train L2: 0.028915 | Test L2: 0.028664
Epoch  200 | Train L2: 0.027277 | Test L2: 0.026496
Epoch  250 | Train L2: 0.025655 | Test L2: 0.024168
Epoch  300 | Train L2: 0.024523 | Test L2: 0.024829
Epoch  350 | Train L2: 0.021642 | Test L2: 0.027381
Epoch  400 | Train L2: 0.020388 | Test L2: 0.023179
Epoch  450 | Train L2: 0.021465 | Test L2: 0.021180
Epoch  500 | Train L2: 0.020645 | Test L2: 0.023799
Epoch  550 | Train L2: 0.021366 | Test L2: 0.020958
Epoch  600 | Train L2: 0.022596 | Test L2: 0.021321
Epoch  650 | Train L2: 0.017997 | Test L2: 0.022262
Epoch  700 | Train L2: 0.017815 | Test L2: 0.018625
Epoch  750 | Train L2: 0.016312 | Test L2: 0.018577
Epoch  800 | Train L2: 0.015689 | Test L2: 0.016458
Epoch  850 | Train L2: 0.016170 | Test L2: 0.016466
Epoch  900 | Train L2: 0.014906 | Test L2: 0.016970
Epoch